# Week 6 homework: some practical aspects of machine learning on graphs.

In this homework, we will implement some tricks that can help improve the effectiveness and efficiency of graph machine learning models. We will be working with the city-roads dataset - a road network of an entire city. The nodes represent road segments, and a directed edge represents that the segments are incident and moving from one to the other is permitted by traffic rules. To start with, let's load the data and see how it is structured.

In [1]:
import yaml
import numpy as np
import pandas as pd

In [2]:
with open('data/city-roads/info.yaml', 'r') as file:
    info = yaml.safe_load(file)

info.keys()

dict_keys(['bin_feature_names', 'cat_feature_names', 'num_feature_names', 'target_name', 'task'])

In [3]:
info['task']

'regression'

In [4]:
info['target_name']

'travel_velocity'

In [5]:
info['num_feature_names']

['length',
 'num_segments',
 'x_coordinate_start',
 'y_coordinate_start',
 'x_coordinate_end',
 'y_coordinate_end']

In [133]:
info['bin_feature_names']

['can_bind_to_reverse_edge',
 'dismount_bike',
 'has_masstransit_lane',
 'ends_with_crosswalk',
 'ends_with_toll_post',
 'is_in_poor_condition',
 'is_paved',
 'is_residential',
 'is_restricted_for_trucks',
 'is_toll',
 'access_0',
 'access_1',
 'access_2',
 'access_3',
 'access_4',
 'access_5']

In [6]:
info['cat_feature_names']

['category', 'edge_type', 'speed_mode', 'speed_limit', 'region_id']

The info dictionary contains some metadata of the dataset - names and types of features, and target name. The task is to predict the average travel velocity on road segments - it is a node regression task. There are three types of node features in the dataset: numerical features, binary features, categorical features.

In [7]:
features_df = pd.read_csv('data/city-roads/features.csv', index_col=0)
features_df.shape

(159894, 28)

In [8]:
features_df.head()

,category,edge_type,speed_mode,speed_limit,region_id,can_bind_to_reverse_edge,dismount_bike,has_masstransit_lane,ends_with_crosswalk,ends_with_toll_post,...,access_3,access_4,access_5,length,num_segments,travel_velocity,x_coordinate_start,y_coordinate_start,x_coordinate_end,y_coordinate_end
0,1,1.0,4.0,6.0,5.0,0.0,0.0,0.0,1.0,0.0,...,1.0,1.0,1.0,12.400000,1,5.879206,60.664605,56.871443,60.664461,56.871522
1,5,1.0,2.0,-1.0,5.0,1.0,0.0,0.0,1.0,0.0,...,1.0,1.0,1.0,9.100000,1,3.226549,60.664605,56.871443,60.664684,56.871512
2,1,1.0,4.0,6.0,5.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,79.199997,2,13.220523,60.664605,56.871443,60.665489,56.870922
3,6,1.0,0.0,-1.0,5.0,1.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,20.200001,1,-1.000000,60.664605,56.871443,60.664401,56.871300
4,5,1.0,2.0,-1.0,5.0,1.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,9.100000,1,0.994184,60.664684,56.871512,60.664605,56.871443


The features_df dataframe contains the features and targets for all nodes. The column names are the same as features and target names in the info dictionary.

In [9]:
edgelist = pd.read_csv('data/city-roads/edgelist.csv').values
edgelist.shape

(403110, 2)

The edgelist array contains edges.

In [10]:
train_mask = pd.read_csv('data/city-roads/train_mask.csv', index_col=0).values.reshape(-1)
val_mask = pd.read_csv('data/city-roads/valid_mask.csv', index_col=0).values.reshape(-1)
test_mask = pd.read_csv('data/city-roads/test_mask.csv', index_col=0).values.reshape(-1)

num_labeled_nodes = train_mask.sum() + val_mask.sum() + test_mask.sum()
print(f'{num_labeled_nodes / len(features_df):.2f} of nodes in the dataset are labeled.')

0.23 of nodes in the dataset are labeled.


The train_mask, val_mask, and test_mask arrays contain the dataset split. Note that not all nodes in the dataset are labeled - for the majority of road segments the average travel velocity is unknown. The unlabeled nodes are the ones for which there is a zero in all three split masks. The target value in the travel_velocity column of features_df is set to -1.

In [11]:
unlabeled_nodes_mask = ~(train_mask | val_mask | test_mask)
assert all(features_df['travel_velocity'].values[unlabeled_nodes_mask] == -1)

Now, let's prepare the features and targets.

In [12]:
from sklearn.preprocessing import OneHotEncoder

In [13]:
targets = features_df['travel_velocity'].values.astype(np.float32)

num_features = features_df[info['num_feature_names']].values.astype(np.float32)
print(f'Numerical features shape: {num_features.shape}')
bin_features = features_df[info['bin_feature_names']].values.astype(np.float32)
print(f'Binary features shape: {bin_features.shape}')
cat_features = features_df[info['cat_feature_names']].values.astype(np.float32)
print(f'Categorical features shape: {cat_features.shape}')
cat_features_one_hot = OneHotEncoder(sparse_output=False, dtype=np.float32).fit_transform(cat_features)
print(f'One-hot encoded categorical features shape: {cat_features_one_hot.shape}')

Numerical features shape: (159894, 6)
Binary features shape: (159894, 16)
Categorical features shape: (159894, 5)
One-hot encoded categorical features shape: (159894, 51)


Binary features do not require any specific preprocessing. We encoded categorical features with one-hot encoding. As for numerical features, it is up to you to preprocess them - we will discuss the options slightly later.

## Part 1: GNN (3 pts).

First, you need to train a GNN for node regression. You can use any architecture, and for implementing it you can use any library: pure PyTorch, DGL, PyG, TensorFlow GNN - whatever you like (does any of you like TensorFlow?). It is up to you to write all the preprocessing, model, and training code - you should already be experienced enough to do it. You can reuse code from the previous seminars and homeworks.

You should use R$^2$ regression metric and the task is to achieve a value of R$^2$ above 0.62 on the test set.

We have some tips for you:

1. Note that the graph as currently stored in the edgelist is **directed** (if you are allowed to move from one road segment to another, it does not always mean that you are allowed to move in the reverse direction). You can either keep the graph this way or transform it to undirected - do whatever you think will work better.
2. Neural networks are not good at working with numerical features with arbitrary scales and distributions (like our current numerical features). It is recommended to preprocess them before passing as input to the model. Scikit-learn has a number of transformations for this - you can read about them [here](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_all_scaling.html#sphx-glr-auto-examples-preprocessing-plot-all-scaling-py). `StandardScaler` provides the simplest transformation, but it is not always the best one, so you might try others as well. You might even choose different transformations for different numerical features. But make sure that you only estimate transformation parameters on train nodes, not val or test ones.
3. It is also often useful to apply some transformation (like scaling) to targets. However, for this particular dataset, it does not seem to improve results, so you do not need to do it. If you still decide to transform the targets and apply some non-linear transformation, make sure that you compute $R^2$ using the original values of the targets, not the transformed ones ($R^2$ is invariant to linear transformations). Transformations from sklearn have `inverse_transform` method which might be useful in this case.
4. A simple mean graph neighborhood aggregation function will probably be enough for your GNN, you do not need to implement anything more complex. But make sure that you are training it for enough steps. And do not forget about skip-connections and layer normalization - they might be very helpful.
5. Do not forget about using mixed precision - it will make your experiments more efficient.

In [14]:
from sklearn.metrics import r2_score
import torch
import torch.nn as nn
import torch_geometric as pyg
from torch_geometric.nn.conv import TransformerConv
import tqdm
from sklearn.preprocessing import StandardScaler, RobustScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau

/home/rsm/VsCode/venv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /home/rsm/VsCode/venv/lib/python3.12/site-packages/libpyg.so: undefined symbol: _ZN5torch8autograd12VariableInfoC1ERKN2at6TensorE
  import torch_geometric.typing
/home/rsm/VsCode/venv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /home/rsm/VsCode/venv/lib/python3.12/site-packages/torch_scatter/_scatter_cuda.so: undefined symbol: _ZN2at4_ops16div__Tensor_mode4callERNS_6TensorERKS2_St8optionalIN3c1017basic_string_viewIcEEE
  import torch_geometric.typing
/home/rsm/VsCode/venv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: /home/rsm/VsCode/venv/lib/python3.12/site-packages/torch_spline_conv/_basis_

In [15]:
std_scaler = StandardScaler()
robust_scaler = RobustScaler()

std_cols = ["x_coordinate_start", "y_coordinate_start", "x_coordinate_end", "y_coordinate_end"]
robust_cols = ["length", "num_segments"]

std_features = np.array(features_df[std_cols])
robust_features = np.array(features_df[robust_cols])

std_scaler.fit(std_features)
robust_scaler.fit(robust_features)

std_features = std_scaler.transform(std_features)
robust_features = robust_scaler.transform(robust_features)

num_features = torch.tensor(np.concatenate((std_features, robust_features), axis=1), dtype=torch.float32)

bin_features = torch.tensor(bin_features, dtype=torch.float32)
cat_features_one_hot = torch.tensor(cat_features_one_hot, dtype=torch.float32)

In [16]:
class FeedForwardModule(nn.Module):
    def __init__(self, dim, dropout, dim_multiplier=4):
        super().__init__()
        
        hidden_dim = dim * dim_multiplier

        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.Dropout(dropout),
            nn.ReLU(),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

class TransformerModule(nn.Module):
    def __init__(self, dim, num_heads, dropout, dim_multiplier):
        super().__init__()
        self.attn = pyg.nn.TransformerConv(
            in_channels=dim,
            out_channels=dim // num_heads,
            heads=num_heads,
            dropout=dropout
        )
        self.ffn = FeedForwardModule(dim, dropout, dim_multiplier)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, edge_index):
        x = x + self.dropout(self.attn(self.norm1(x), edge_index))
        x = self.norm2(self.ffn(x))
        return x

class GraphTransformer(nn.Module):
    def __init__(self,
                 num_layers: int,
                 hidden_dim: int,
                 num_heads: int,
                 hidden_dim_multiplier: int,
                 dropout: float,
                 input_dim: int):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
        )

        self.layers = nn.ModuleList([
            TransformerModule(hidden_dim, num_heads, dropout, hidden_dim_multiplier)
            for _ in range(num_layers)
        ])

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, edge_index):
        x = self.input_proj(x)

        for layer in self.layers:
            x = layer(x, edge_index)

        output = self.head(x)

        return output.squeeze(1)

In [17]:
X = torch.cat([num_features, bin_features, cat_features_one_hot], dim=1)
y = torch.tensor(targets)
input_dim = X.shape[1]

edge_index = torch.tensor(edgelist).transpose(0, 1)
def to_numpy(x):
    return x.detach().cpu().numpy()

In [18]:
model = GraphTransformer(num_layers=2, hidden_dim=160, num_heads=8, hidden_dim_multiplier=4, dropout=0.1, input_dim=input_dim)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
#scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=25, min_lr=2e-5)

In [19]:
def train(edge_index, n_steps, path):
    max_r2 = 0

    for step in range(n_steps):
        model.train()
        preds = model(X, edge_index)

        loss = criterion(preds[train_mask], y[train_mask])

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        model.eval()
        with torch.no_grad():
            preds = model(X, edge_index)
            y_val, preds_val = to_numpy(y[val_mask]), to_numpy(preds[val_mask])
            r2 = r2_score(y_true=y_val, y_pred=preds_val)
            if (r2 > max_r2):
                max_r2 = r2
                torch.save({
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "step": step,
                    "max_r2": max_r2,
                }, path)

        print(f"Step: {step} | Loss: {loss.item()} | R2: {r2}")

In [20]:
def test(path):
    checkpoint = torch.load(path)

    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])

    model.eval()
    with torch.no_grad():
        preds = model(X, edge_index)
        y_test = to_numpy(y[test_mask])
        preds_test = to_numpy(preds[test_mask])

    print(f"Final R2 score on test: {r2_score(y_true=y_test, y_pred=preds_test)}")

In [ ]:
train(edge_index=edge_index, n_steps=400, path="checkpoint1.pt")

In [21]:
model = GraphTransformer(num_layers=2, hidden_dim=160, num_heads=8, hidden_dim_multiplier=4, dropout=0.1, input_dim=input_dim)
test("checkpoint.pt")

Final R2 score on test: 0.6318771839141846


## Part 2: additional node features - DeepWalk embeddings (1 pt).

Our graph represents a road network, and thus can be naturally embedded in 2D space. For such graphs, additional information about the global positions of nodes can provide useful information. This information is already partly present in the data, since it contains coordinates of the start and the end of each road segment in the node features, but we can try providing even more information by augmenting node features with DeepWalk embeddings. Train DeepWalk embeddings for our graph, add them to node features, and train the GNN from the previous part using these augmented features. You should get a value of R$^2$ on the test set at least 0.01 higher than in the previous part.

[This](https://docs.dgl.ai/en/1.1.x/generated/dgl.nn.pytorch.DeepWalk.html) module and its example usage might be helpful, but you probably want to move the training to GPU so it is faster. Note that you do not need a particularly high embedding dimension in this task - 64 will be enough. And of course, you are free to use any other implementation of DeepWalk if you want to.

In [22]:
from torch_geometric.utils import add_self_loops
num_nodes = len(features_df)
edge_index_with_loops, _ = add_self_loops(edge_index, num_nodes=num_nodes)

deepwalk_emb_dim = 64
walk_lenght = 60
context_size = 6
walks_per_node=10
num_negative_samples = 10

In [ ]:
import torch
from torch_geometric.nn import Node2Vec

deep_walk_model = Node2Vec(
    edge_index=edge_index_with_loops,
    embedding_dim=deepwalk_emb_dim,
    walk_length=walk_lenght,
    context_size=context_size,
    walks_per_node=walks_per_node,
    num_negative_samples=num_negative_samples,
    p=1.0,
    q=1.0,
)

loader = deep_walk_model.loader(batch_size=256, shuffle=True, num_workers=4)

optimizer = torch.optim.Adam(deep_walk_model.parameters(), lr=0.01)


for epoch in range(3):
    for pos_rw, neg_rw in loader:
        optimizer.zero_grad()
        loss = deep_walk_model.loss(pos_rw, neg_rw)
        loss.backward()
        optimizer.step()
    print(f"Epoch{epoch} | Loss: {loss.item()}")

deep_walk_embeddings = deep_walk_model.embedding.weight

torch.save(deep_walk_embeddings, f"embeddings_{walk_lenght}_{context_size}_{walks_per_node}_{num_negative_samples}.pt")

In [23]:
deep_walk_embeddings = torch.load(f"embeddings_{walk_lenght}_{context_size}_{walks_per_node}_{num_negative_samples}.pt")
deep_walk_embeddings = deep_walk_embeddings.clone().detach()

In [24]:
X = torch.cat([num_features, bin_features, cat_features_one_hot, deep_walk_embeddings], dim=1)
input_dim = X.shape[1]

model = GraphTransformer(num_layers=2, hidden_dim=160, num_heads=8, hidden_dim_multiplier=4, dropout=0.1, input_dim=input_dim)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

In [ ]:
train(edge_index=edge_index, n_steps=400, path="checkpoint_deepwalk1.pt")

In [25]:
test("checkpoint_deepwalk.pt")

Final R2 score on test: 0.6563677191734314


## Part 3: graph-agnostic model (2 pts).

Sometimes the graph is so large you cannot fit it into memory. In this case, you can try minibatch training of a GNN on subgraphs, distributed training of a GNN, or train a graph-agnostic model on minibatches of nodes treating them as independent samples. The last approach is what you need to implement in this part. Create a graph-agnostic model (i.e., a model that does not use the graph structure) and a standard training loop that uses minibatches of samples (assumed to be independent) instead of full graph. In this part, the model should be a neural network (e.g., an MLP or a ResNet). Train this model first using only the original node features, and then using node features augmented with DeepWalk embeddings. The DeepWalk embeddings in the second experiment will provide the model with some information about the graph. We will see in later parts how to design even more graph-based features for our graph-agnostic model to push its results further.

In this part, you should achieve a value of R$^2$ on the test set above 0.54 in the first experiment (with only original node features) and above 0.62 in the second experiment (with DeepWalk feature augmentation). Note that DeepWalk embeddings help the graph-agnostic model much more than they help the GNN, because for the graph-agnostic model they are the only source of information about the graph.

In [34]:
class ResidualBlock(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(input_size, hidden_size) if input_size != hidden_size else nn.Identity()
        self.rel = nn.ReLU()

    def forward(self, x):
        out = self.fc1(x)
        out = self.rel(out)
        out = self.fc2(out)
        out = out + self.fc3(x)
        out = self.rel(out)
        return out
    
class AgnosticModel(nn.Module):
    def __init__(self, input_dim, sizes):
        super().__init__()


        self.blocks = nn.ModuleList(
            [ResidualBlock(input_size=input_dim, hidden_size=sizes[0])] + 
            [ResidualBlock(input_size=input_size, hidden_size=hidden_size) for input_size, hidden_size in zip(sizes, sizes[1:])]
        )

        self.head = nn.Linear(sizes[-1], 1)

    def forward(self, x):

        for block in self.blocks:
            x = block(x)

        output = self.head(x)

        return output.squeeze(1)

In [35]:
from torch.utils.data import Dataset, DataLoader

class RoadDataset(Dataset):
    def __init__(self, X, y):
        super().__init__()
        self.X = X
        self.y = y
    
    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [36]:
def train_with_batches(n_steps, path):

    train_dataset = RoadDataset(X[train_mask], y[train_mask])
    val_dataset   = RoadDataset(X[val_mask], y[val_mask])

    train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=1024, shuffle=False)

    max_r2 = 0
    for step in range(n_steps):
        model.train()
        train_loss = 0

        for xb, yb in train_loader:
            preds = model(xb)
            loss = criterion(preds, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * xb.size(0)

        train_loss /= len(train_dataset)

        model.eval()
        val_preds = []
        val_true  = []

        with torch.no_grad():
            for xb, yb in val_loader:
                pred = model(xb)
                val_preds.append(pred.cpu())
                val_true.append(yb.cpu())

        val_preds = torch.cat(val_preds).numpy()
        val_true  = torch.cat(val_true).numpy()

        r2 = r2_score(val_true, val_preds)

        if (r2 > max_r2):
            max_r2 = r2
            torch.save({
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "step": step,
                "max_r2": max_r2,
            }, path)

        print(f"Step {step} | Train loss: {train_loss:.4f} | R2: {r2:.4f}")

In [37]:
def test_with_batches(path):
    test_dataset   = RoadDataset(X[val_mask], y[val_mask])
    test_loader   = DataLoader(test_dataset, batch_size=1024, shuffle=False)

    checkpoint = torch.load(path)

    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])

    model.eval()
    test_preds = []
    test_true  = []

    with torch.no_grad():
        for xb, yb in test_loader:
            preds = model(xb)
            test_preds.append(preds.cpu())
            test_true.append(yb.cpu())

    test_preds = torch.cat(test_preds).numpy()
    test_true  = torch.cat(test_true).numpy()


    print(f"Final R2 score on test: {r2_score(y_true=test_true, y_pred=test_preds)}")

In [38]:
X = torch.cat([num_features, bin_features, cat_features_one_hot], dim=1)
input_dim = X.shape[1]

model = AgnosticModel(input_dim=input_dim, sizes=[256, 256])
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

In [39]:
train_with_batches(n_steps=100, path="checkpoint_agnostic.pt")

Step 0 | Train loss: 56.0277 | R2: -0.7820
Step 1 | Train loss: 22.9607 | R2: 0.0543
Step 2 | Train loss: 16.3515 | R2: 0.2135
Step 3 | Train loss: 13.7788 | R2: 0.3250
Step 4 | Train loss: 12.0377 | R2: 0.3793
Step 5 | Train loss: 11.1480 | R2: 0.4124
Step 6 | Train loss: 10.5194 | R2: 0.4370
Step 7 | Train loss: 10.0199 | R2: 0.4588
Step 8 | Train loss: 9.5805 | R2: 0.4836
Step 9 | Train loss: 9.2394 | R2: 0.5015
Step 10 | Train loss: 8.9907 | R2: 0.5115
Step 11 | Train loss: 8.8222 | R2: 0.5158
Step 12 | Train loss: 8.6691 | R2: 0.5213
Step 13 | Train loss: 8.5697 | R2: 0.5305
Step 14 | Train loss: 8.4915 | R2: 0.5325
Step 15 | Train loss: 8.4228 | R2: 0.5332
Step 16 | Train loss: 8.3977 | R2: 0.5272
Step 17 | Train loss: 8.3221 | R2: 0.5416
Step 18 | Train loss: 8.2421 | R2: 0.5435
Step 19 | Train loss: 8.1577 | R2: 0.5451
Step 20 | Train loss: 8.1112 | R2: 0.5447
Step 21 | Train loss: 8.0659 | R2: 0.5498
Step 22 | Train loss: 8.0013 | R2: 0.5527
Step 23 | Train loss: 7.9503 | R2: 

In [40]:
model = AgnosticModel(input_dim=input_dim, sizes=[256, 256])
test_with_batches("checkpoint_agnostic.pt")

Final R2 score on test: 0.6017032861709595


In [42]:
X = torch.cat([num_features, bin_features, cat_features_one_hot, deep_walk_embeddings], dim=1)
input_dim = X.shape[1]

model = AgnosticModel(input_dim=input_dim, sizes=[256, 256])
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

train_with_batches(n_steps=100, path="checkpoint_agnostic_deepwalks.pt")

Step 0 | Train loss: 53.9356 | R2: -0.6782
Step 1 | Train loss: 21.0992 | R2: 0.1272
Step 2 | Train loss: 15.5503 | R2: 0.2493
Step 3 | Train loss: 13.4452 | R2: 0.3322
Step 4 | Train loss: 11.8239 | R2: 0.3897
Step 5 | Train loss: 10.7940 | R2: 0.4247
Step 6 | Train loss: 10.0987 | R2: 0.4521
Step 7 | Train loss: 9.5808 | R2: 0.4702
Step 8 | Train loss: 9.1735 | R2: 0.4951
Step 9 | Train loss: 8.8043 | R2: 0.5124
Step 10 | Train loss: 8.4847 | R2: 0.5259
Step 11 | Train loss: 8.2181 | R2: 0.5379
Step 12 | Train loss: 7.9968 | R2: 0.5481
Step 13 | Train loss: 7.7948 | R2: 0.5548
Step 14 | Train loss: 7.5900 | R2: 0.5648
Step 15 | Train loss: 7.4035 | R2: 0.5708
Step 16 | Train loss: 7.2053 | R2: 0.5713
Step 17 | Train loss: 7.0141 | R2: 0.5878
Step 18 | Train loss: 6.8003 | R2: 0.5927
Step 19 | Train loss: 6.5928 | R2: 0.6020
Step 20 | Train loss: 6.3798 | R2: 0.6078
Step 21 | Train loss: 6.1622 | R2: 0.6113
Step 22 | Train loss: 5.9348 | R2: 0.6205
Step 23 | Train loss: 5.7515 | R2: 0

In [43]:
model = AgnosticModel(input_dim=input_dim, sizes=[256, 256])
test_with_batches("checkpoint_agnostic_deepwalks.pt")

Final R2 score on test: 0.6639426946640015


## Part 4: additional node features - neighborhood feature aggregation (1 pts).

As we have seen in the previous part, DeepWalk embeddings are very helpful for graph-agnostic models. However, these embeddings can only directly provide information about the global position of nodes in the graph. Can we additionally provide our graph-agnostic model with information about node features in the graph neighborhood of each node? Yes, we can. In this part, you should augment node features with statistics computed over one-hop neighborhood features. For numerical node features, compute their mean, maximum, and minimum values over the neighborhood. For binary and one-hot encoded categorical features, compute just their mean values over the neighborhood.

Note that information about neighborhood features that we provide for our graph-agnostic model in this task is exactly the kind of information that GNNs use, but GNNs can process it in trickier ways and aggregate it over larger neighborhoods.

Further, add node degrees to node features. It will provide some more information about the graph structure to your graph-agnostic model. It is useful to divide node degree values by their maximum value to scale them into the \[0; 1\] interval, this way it will be easier for neural networks to work with them.

Run an experiment that trains your graph-agnostic model on the combination of original node features, DeepWalk embeddings, aggregated node features, and a node degree feature. You should get a value of R$^2$ on the test set at least 0.01 higher than in the previous part.

In [44]:
neighbors = [[] for _ in range(num_nodes)]

for u, v in edgelist:
    neighbors[u].append(v)
    neighbors[v].append(u)

In [45]:
num_stats = [[] for _ in range(num_nodes)]
bin_stats = [[] for _ in range(num_nodes)]
one_hot_stats = [[] for _ in range(num_nodes)]

for i in range(num_nodes):
    neighbors_num = torch.stack([num_features[j] for j in neighbors[i]])
    num_stats[i] = torch.cat(
        [neighbors_num.mean(dim=0), neighbors_num.min(dim=0).values, neighbors_num.max(dim=0).values],
        dim=0
    )

    neighbors_bin = torch.stack([bin_features[j] for j in neighbors[i]])
    bin_stats[i] = neighbors_bin.mean(dim=0)

    neighbors_one_hot = torch.stack([cat_features_one_hot[j] for j in neighbors[i]])
    one_hot_stats[i] = neighbors_one_hot.mean(dim=0)

num_stats = torch.stack(num_stats)
bin_stats = torch.stack(bin_stats)
one_hot_stats = torch.stack(one_hot_stats)

In [93]:
X = torch.cat([
    num_features, bin_features, cat_features_one_hot, deep_walk_embeddings,
    num_stats, bin_stats, one_hot_stats               
], dim=1)
input_dim = X.shape[1]

model = AgnosticModel(input_dim=input_dim, sizes=[256, 256])
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

In [94]:
train_with_batches(n_steps=50, path="checkpoint_agnostic_deepwalks_stats.pt")

Step 0 | Train loss: 60.1937 | R2: -1.3263
Step 1 | Train loss: 27.6635 | R2: -0.0467
Step 2 | Train loss: 16.9017 | R2: 0.1774
Step 3 | Train loss: 14.3258 | R2: 0.2746
Step 4 | Train loss: 12.6074 | R2: 0.3497
Step 5 | Train loss: 11.3871 | R2: 0.4012
Step 6 | Train loss: 10.5220 | R2: 0.4347
Step 7 | Train loss: 9.9662 | R2: 0.4563
Step 8 | Train loss: 9.5362 | R2: 0.4754
Step 9 | Train loss: 9.2048 | R2: 0.4927
Step 10 | Train loss: 8.8540 | R2: 0.5063
Step 11 | Train loss: 8.5723 | R2: 0.5233
Step 12 | Train loss: 8.2940 | R2: 0.5368
Step 13 | Train loss: 8.0538 | R2: 0.5415
Step 14 | Train loss: 7.8364 | R2: 0.5582
Step 15 | Train loss: 7.6576 | R2: 0.5668
Step 16 | Train loss: 7.5079 | R2: 0.5734
Step 17 | Train loss: 7.4101 | R2: 0.5761
Step 18 | Train loss: 7.2464 | R2: 0.5850
Step 19 | Train loss: 7.1253 | R2: 0.5887
Step 20 | Train loss: 7.0644 | R2: 0.5922
Step 21 | Train loss: 6.9314 | R2: 0.5987
Step 22 | Train loss: 6.8033 | R2: 0.6034
Step 23 | Train loss: 6.7019 | R2: 

In [95]:
test_with_batches("checkpoint_agnostic_deepwalks_stats.pt")

Final R2 score on test: 0.6805593967437744


## Part 5: additional node features - neighborhood target aggregation (1 pt).

In the previous part, we augmented node features with statistics of features of their neighbors. Can we also add statistics of targets of their neighbors? Yes, we can. Note that you cannot straightforwardly do it for GNNs, since they know about edges in the graph and can recover node targets from features used for their neighbors. But graph-agnostic models treat all nodes as independent samples, and thus adding targets of neighborhood nodes to a node's features will not lead to any problems.

**Important:** note that when computing statistics of neighborhood targets, you can only consider nodes from the train set. **Do not** compute statistics over val, test, and unlabeled nodes.

However, for our particular dataset there is a problem with straightforwardly applying this approach. Do you remember how many labeled nodes we have?

Most nodes in the graph do not have even a single train node among their neighbors. What can we do about this? Let's consider larger neighborhoods. In this task, you should compute target statistics over each node's multihop neighborhood. The number of hops is a hyperperameter that you can experiment with, but 5 works fine. Think how you can construct the multihop neighborhoods efficiently.

Compute the mean of targets of train nodes in each node's multihop neighborhood and add it to node features. For nodes that do not have any train nodes even in their multihop neighborhood, set the new feature value to the mean of all its values for those nodes for which the set of train nodes in the multihop neighborhood is non-empty. Add a separate binary feature that tells if the statistic was actually computed over the node neighborhood or just filled with the mean (i.e., the value 1 of this feature means that there are no train nodes in the multihop neighborhood of this node).

Run an experiment that trains your graph-agnostic model on the combination of original node features, DeepWalk embeddings, aggregated node features, a node degree feature, aggregated target features, and a binary feature discussed above. You should get a value of R$^2$ on the test set at least 0.01 higher than in the previous part.

In [96]:
from collections import deque

def compute_average_local_target(num_hops):
    locals = [[] for _ in range(num_nodes)]

    for i in range(num_nodes):
        visited = set()
        visited.add(i)
        queue = deque([(i, 0)])

        while queue:
            node, dist = queue.popleft()

            if dist == num_hops:
                continue

            for j in neighbors[node]:
                if j not in visited:
                    visited.add(j)
                    queue.append((j, dist + 1))
                    
        locals[i] = list(visited)

    locals_y = [torch.nan for i in range(num_nodes)]
    for i in range(num_nodes):
        locals_y[i] = torch.tensor(
            [y[j] for j in locals[i] if train_mask[j]],
            dtype=torch.float32
        ).mean()

    locals_y = torch.tensor(locals_y, dtype= torch.float32)

    mask_locals_nan = torch.tensor(torch.isnan(locals_y), dtype=torch.float32)

    locals_y = torch.nan_to_num(locals_y, nan=torch.nanmean(locals_y))
    locals_y = locals_y.unsqueeze(1)
    mask_locals_nan = mask_locals_nan.unsqueeze(1)

    return locals_y, mask_locals_nan

In [98]:
locals_y, mask_locals_nan = compute_average_local_target(num_hops=5)

/tmp/ipykernel_170168/436157944.py:33: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask_locals_nan = torch.tensor(torch.isnan(locals_y), dtype=torch.float32)


In [102]:
X = torch.cat([
    num_features, bin_features, cat_features_one_hot, deep_walk_embeddings,
    num_stats, bin_stats, one_hot_stats,
    locals_y, mask_locals_nan               
], dim=1)
input_dim = X.shape[1]

model = AgnosticModel(input_dim=input_dim, sizes=[400, 400])
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

In [103]:
train_with_batches(n_steps=100, path="checkpoint_agnostic_deepwalks_stats_targets.pt")

Step 0 | Train loss: 34.5215 | R2: 0.0516
Step 1 | Train loss: 11.4971 | R2: 0.4637
Step 2 | Train loss: 8.6384 | R2: 0.5323
Step 3 | Train loss: 7.3329 | R2: 0.5681
Step 4 | Train loss: 6.8330 | R2: 0.5823
Step 5 | Train loss: 6.5350 | R2: 0.5970
Step 6 | Train loss: 6.3255 | R2: 0.6097
Step 7 | Train loss: 6.1389 | R2: 0.6193
Step 8 | Train loss: 5.9965 | R2: 0.6279
Step 9 | Train loss: 5.8689 | R2: 0.6367
Step 10 | Train loss: 5.8046 | R2: 0.6426
Step 11 | Train loss: 5.6891 | R2: 0.6468
Step 12 | Train loss: 5.6077 | R2: 0.6525
Step 13 | Train loss: 5.5428 | R2: 0.6530
Step 14 | Train loss: 5.4471 | R2: 0.6595
Step 15 | Train loss: 5.3687 | R2: 0.6618
Step 16 | Train loss: 5.2638 | R2: 0.6663
Step 17 | Train loss: 5.1915 | R2: 0.6706
Step 18 | Train loss: 5.1091 | R2: 0.6587
Step 19 | Train loss: 5.0183 | R2: 0.6749
Step 20 | Train loss: 4.8664 | R2: 0.6709
Step 21 | Train loss: 4.7437 | R2: 0.6828
Step 22 | Train loss: 4.6238 | R2: 0.6807
Step 23 | Train loss: 4.5158 | R2: 0.6856


In [104]:
test_with_batches("checkpoint_agnostic_deepwalks_stats_targets.pt")

Final R2 score on test: 0.6952773332595825


## Part 6: GBDT (2 pts).

The dataset contains important numerical node features. Neural networks are generally not very good at processing such features (even with appropriate transformations), and decision tree based models, such as random forest or gradient-boosted decision trees (GBDT), can often achieve better results on such data. In this part, you need to train a graph-agnostic GBDT model on our dataset. Use all the node features from the previous part (including graph-based ones). You should achieve a value of R$^2$ on the test set above 0.72.

There are three most popular GBDT implementations: XGBoost, LightGBM, and CatBoost. You can use any of them, but we recommend LightGBM for this part, because it is the fastest one (and you might need to do quite a few training runs for hyperparameter tuning).

In [105]:
y = y.numpy()

In [106]:
from catboost import CatBoostRegressor, Pool
import pandas as pd

cat_features = torch.tensor(cat_features, dtype=torch.float32)

X = torch.cat([
    num_features, bin_features, cat_features_one_hot, deep_walk_embeddings,
    num_stats, bin_stats, one_hot_stats,
    locals_y, mask_locals_nan               
], dim=1).numpy()

train_pool = Pool(X, y)
test_pool = Pool(X[val_mask], y[val_mask])

model = CatBoostRegressor(
    iterations=10000,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    verbose=100
)

model.fit(train_pool, eval_set=test_pool)

0:	learn: 3.8383688	test: 7.2498636	best: 7.2498636 (0)	total: 67.2ms	remaining: 11m 11s
100:	learn: 1.7841346	test: 3.0738418	best: 3.0738418 (100)	total: 2.08s	remaining: 3m 23s
200:	learn: 1.7179775	test: 2.9519111	best: 2.9519111 (200)	total: 3.99s	remaining: 3m 14s
300:	learn: 1.6786252	test: 2.8890447	best: 2.8890447 (300)	total: 5.87s	remaining: 3m 9s
400:	learn: 1.6426144	test: 2.8337918	best: 2.8337918 (400)	total: 8.07s	remaining: 3m 13s
500:	learn: 1.6105518	test: 2.7828563	best: 2.7828563 (500)	total: 10.6s	remaining: 3m 21s
600:	learn: 1.5821330	test: 2.7391262	best: 2.7391262 (600)	total: 13s	remaining: 3m 23s
700:	learn: 1.5563250	test: 2.6991809	best: 2.6991809 (700)	total: 15.1s	remaining: 3m 20s
800:	learn: 1.5319243	test: 2.6589343	best: 2.6589343 (800)	total: 17.1s	remaining: 3m 16s
900:	learn: 1.5104553	test: 2.6214294	best: 2.6214294 (900)	total: 19.5s	remaining: 3m 17s
1000:	learn: 1.4918830	test: 2.5888233	best: 2.5888233 (1000)	total: 21.6s	remaining: 3m 13s
11

In [107]:
y_pred = model.predict(X[test_mask])
r2 = r2_score(y_pred, y[test_mask])

print("R2 score:", r2)

R2 score: 0.8809217203681011


## Bonus part (up to 3 pts).

You have seen that GBDT can achieve very strong results on our dataset. But can you make a neural network achieve almost equally strong results? In this part, you can try to train a neural model to achive a test value of R$^2$ above 0.70. You can use any tricks and modify node features, model, and training process in any way. You will get 1 point if your test R$^2$ is at least 0.70, 1 more point if your test R$^2$ is at least 0.71, and 1 more point if your test R$^2$ is at least 0.72. **To get the points, besides training the model, you need to describe below what you did to achieve these results.**